# Next-Day Prediction Pre-training for SwinUnet

Self-supervised pre-training: predict tomorrow's dynamic features (weather, vegetation,
forecasts) from today's full feature set. The model learns spatial weather evolution
across terrain — the same spatial propagation skill needed for fire spread prediction.

**Loss is computed only on dynamic channels** (17 of 40) that actually change day-to-day.
Static features (terrain, landcover) are visible as input context but not predicted.

**Prerequisites:**
- Pre-training TIFs with 2-day pairs in GCS bucket
- **Runtime -> Change runtime type -> GPU** (T4 or better)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
REPO_ORG   = "amindell11"
REPO_NAME  = "wildfire-ts-swin"
REPO_BRANCH = "feature/nextday-pretrain"

GCS_BUCKET  = "lmudl-wildfire-compilation-bucket"
GCS_PREFIX  = "WildfireSpreadTS/pretrain"
LOCAL_TIF_DIR = "/content/pretrain_tifs"

OUTPUT_DIR  = "/content/drive/MyDrive/wildfire_runs/pretrain_nextday"

MAX_EPOCHS          = 200
BATCH_SIZE          = 32
BASE_LR             = 1.5e-4
WEIGHT_DECAY        = 0.05
WARMUP_EPOCHS       = 10
CROP_SIDE_LENGTH    = 128
CROPS_PER_LOCATION  = 4
STATS_YEARS         = (2020, 2021)
NUM_WORKERS         = 4
SEED                = 42

CHECKPOINT_INTERVAL = 20

## 3. Download TIFs from GCS

Each location folder should contain 2 TIF files (consecutive days).

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
os.makedirs(LOCAL_TIF_DIR, exist_ok=True)

!gsutil -m cp -r gs://{GCS_BUCKET}/{GCS_PREFIX}/* {LOCAL_TIF_DIR}/

# Count locations with 2-day pairs
import glob
all_tifs = glob.glob(f"{LOCAL_TIF_DIR}/**/*.tif", recursive=True)
locations = set(os.path.dirname(t) for t in all_tifs)
pairs = sum(1 for loc in locations if len(glob.glob(f"{loc}/*.tif")) >= 2)
print(f"Downloaded {len(all_tifs)} TIFs across {len(locations)} locations")
print(f"Locations with 2-day pairs: {pairs}")

## 4. Clone repo and install dependencies

In [ ]:
_repo_url = f"https://github.com/{REPO_ORG}/{REPO_NAME}.git"
!rm -rf /content/{REPO_NAME}
!git clone -b {REPO_BRANCH} {_repo_url} /content/{REPO_NAME}
!pip install -q -r /content/{REPO_NAME}/requirements.txt
!pip install -q rasterio
!git -C /content/{REPO_NAME} log --oneline -5

In [ ]:
import sys
sys.path.insert(0, f'/content/{REPO_NAME}')

## 5. Detect accelerator

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("WARNING: No GPU detected.")

## 6. Build Model & Preview

In [ ]:
import os
import random
import types
import numpy as np
import torch
import torch.backends.cudnn as cudnn

from config import get_config
from networks.nextday_swin_unet import NextDaySwinUnet, DYNAMIC_CHANNELS, N_DYNAMIC
from trainer_nextday_pretrain import trainer_nextday_pretrain
from datasets.wildfire.pretrain_dataset import PretrainDataset

if torch.cuda.is_available():
    cudnn.benchmark = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

config_args = types.SimpleNamespace(
    cfg=f'/content/{REPO_NAME}/configs/swin_tiny_patch4_window4_128_wildfire.yaml',
    opts=['MODEL.SWIN.IN_CHANS', '40', 'MODEL.SWIN.N_TIMESTEPS', '1', 'MODEL.PRETRAIN_CKPT', 'None'],
    batch_size=None, zip=False, cache_mode='part', resume=None,
    accumulation_steps=None, use_checkpoint=False, amp_opt_level='',
    tag=None, eval=False, throughput=False,
)
config = get_config(config_args)

model = NextDaySwinUnet(
    config, img_size=CROP_SIDE_LENGTH, use_factored_embed=False,
).to(DEVICE)

dataset = PretrainDataset(
    tif_dir=LOCAL_TIF_DIR,
    crop_side_length=CROP_SIDE_LENGTH,
    stats_years=STATS_YEARS,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Predicting {N_DYNAMIC} dynamic channels from 40 input channels")
print(f"Location pairs with 2 days: {len(dataset)}")

In [ ]:
import matplotlib.pyplot as plt

# Preview: one sample showing today / actual tomorrow / predicted tomorrow
VIS_FEATURES = {
    'NDVI':      (3,  DYNAMIC_CHANNELS.index(3)),
    'Precip':    (5,  DYNAMIC_CHANNELS.index(5)),
    'Wind Spd':  (6,  DYNAMIC_CHANNELS.index(6)),
    'Temp Max':  (9,  DYNAMIC_CHANNELS.index(9)),
    'Fcst Temp': (36, DYNAMIC_CHANNELS.index(36)),
}

x_day1, y_day2 = dataset[0]
x_day1 = x_day1.unsqueeze(0).to(DEVICE)
y_day2 = y_day2.unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    loss, pred = model(x_day1, y_day2)

x_cpu = x_day1[0].cpu()
y_cpu = y_day2[0].cpu()
p_cpu = pred[0].cpu()

def normalize_for_display(t):
    lo, hi = t.min(), t.max()
    if hi - lo < 1e-8:
        return torch.zeros_like(t)
    return (t - lo) / (hi - lo)

n_ch = len(VIS_FEATURES)
fig, axes = plt.subplots(3, n_ch, figsize=(3.5 * n_ch, 9))

row_labels = ['Today (input)', 'Tomorrow (actual)', 'Tomorrow (predicted,\nuntrained)']
for col, (name, (input_ch, pred_ch)) in enumerate(VIS_FEATURES.items()):
    imgs = [
        normalize_for_display(x_cpu[input_ch]),
        normalize_for_display(y_cpu[input_ch]),
        normalize_for_display(p_cpu[pred_ch]),
    ]
    for row in range(3):
        ax = axes[row, col]
        ax.imshow(imgs[row].numpy(), cmap='viridis', vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(name, fontsize=10)
        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=9)

plt.suptitle(f'Next-Day Prediction Preview (untrained, loss={loss.item():.3f})', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"After training, row 3 should closely match row 2.")

## 7. Run Pre-training

Automatically resumes from the latest checkpoint if one exists.  
Set `NO_RESUME = True` to train from scratch.

In [ ]:
NO_RESUME = False

%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/pretrain_log

args = types.SimpleNamespace(
    tif_dir=LOCAL_TIF_DIR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
    warmup_epochs=WARMUP_EPOCHS,
    num_workers=NUM_WORKERS,
    crop_side_length=CROP_SIDE_LENGTH,
    stats_years=STATS_YEARS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    no_resume=NO_RESUME,
    crops_per_location=CROPS_PER_LOCATION,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer_nextday_pretrain(args, model, OUTPUT_DIR, device=DEVICE)

## 8. Next steps

Pre-trained weights are saved to Google Drive:
- `{OUTPUT_DIR}/pretrain_best.pth` -- best val loss (encoder+decoder only)
- `{OUTPUT_DIR}/pretrain_encoder_decoder.pth` -- final epoch

To fine-tune on the fire dataset, open `train_and_test_colab.ipynb` and set:
```python
PRETRAIN_WEIGHTS = "/content/drive/MyDrive/wildfire_runs/pretrain_nextday/pretrain_best.pth"
```